# Environment

In [ ]:
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

from kbase.config import settings
import kbase.preprocessing as dp 

# Data Loading & Filtering

In [ ]:
# Load general preprocessed triage table
df_triaje = pq.read_table(
    os.path.join(settings.source_tables_path, settings.triaje_table_cleaned_path)
).to_pandas()

# Create general stroke table filtered by stroke patients and valid values in outcome variable 
df_stroke_general = df_triaje[df_triaje["demand_type_1"] == 54]
df_stroke_general = df_stroke_general.dropna(subset=["p1_real_emerg"])

# Load reduced stroke table (exported from modelling.py)
df_stroke_reduced = pq.read_table(
    os.path.join(settings.source_tables_path, settings.stroke_table_cleaned_path)
).to_pandas()
print(f"Shape: {df_stroke_reduced.shape}")
df_stroke_reduced.head(3)

# ATC Medication Columns — Completeness Analysis

In [ ]:
df_stroke_general = dp.analyze_atc_columns(
    df_stroke_general,
    threshold=1.0,
    show_counts=True,
    show_histogram=True,
    export=False,
    export_path=os.path.join(settings.source_tables_path, settings.stroke_table_cleaned_path),
    drop_columns=True,
)


# Correlation Analysis by Feature Group

In [ ]:
# ---------------------------------------------------------------------------
# Correlation analysis by feature group
# Triage questions (digit prefix) are excluded — independent by design
# ---------------------------------------------------------------------------

# --- Column groups ---
triage_cols = [c for c in df_stroke_reduced.columns if c[0].isdigit()]
lr_cols     = [c for c in df_stroke_reduced.columns if c.startswith("lr_")]
hist_cols   = [c for c in df_stroke_reduced.columns if c.startswith("hist_")]
atc_cols    = [c for c in df_stroke_reduced.columns if c.startswith("atc")]
other_cols  = [
    c for c in df_stroke_reduced.columns
    if c not in triage_cols + lr_cols + hist_cols + atc_cols
]

print("=" * 40)
print("FEATURE SUMMARY BY GROUP")
print("=" * 40)
print(f"  Triage questions (digit):  {len(triage_cols):>4}  (excluded from correlation)")
print(f"  Literal reason   (lr_):    {len(lr_cols):>4}")
print(f"  Past history     (hist_):  {len(hist_cols):>4}")
print(f"  Medication       (atc):    {len(atc_cols):>4}")
print(f"  Other:                     {len(other_cols):>4}")
print(f"  {'─'*30}")
print(f"  Total features:            {len(df_stroke_reduced.columns):>4}")
print("=" * 40)


def _corr_analysis(
    df, cols, group_name,
    method="pearson",
    threshold=0.3,
    top_n=20,
    max_heatmap_cols=60,
):
    """
    Pearson correlation matrix for a feature group.

    - Full heatmap when cols <= max_heatmap_cols.
    - Filtered heatmap (only columns involved in high-correlation pairs)
      when the group is larger, capped at max_heatmap_cols.
    - Always prints the top_n pairs sorted by |r|.
    """
    data = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .dropna(how="all", axis=1)
    )
    if data.shape[1] < 2:
        print(f"\n{group_name}: not enough valid columns to correlate.")
        return

    corr = data.corr(method=method)

    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["var1", "var2", "r"]
    pairs["abs_r"] = pairs["r"].abs()
    pairs = pairs.sort_values("abs_r", ascending=False)
    high_corr = pairs[pairs["abs_r"] >= threshold]

    print(f"\n{'='*60}")
    print(f"  {group_name}  ({data.shape[1]} features, method={method})")
    print(f"  Pairs with |r| >= {threshold}: {len(high_corr)}")
    print(f"{'='*60}")

    if data.shape[1] <= max_heatmap_cols:
        plot_cols = data.columns.tolist()
        subtitle  = ""
    else:
        involved  = set(high_corr["var1"]) | set(high_corr["var2"])
        plot_cols = [c for c in data.columns if c in involved][:max_heatmap_cols]
        subtitle  = f"  (columns with |r| >= {threshold}, capped at {max_heatmap_cols})"

    if len(plot_cols) >= 2:
        corr_plot = data[plot_cols].corr(method=method)
        n = len(plot_cols)
        size = max(6, n * 0.4)
        fig, ax = plt.subplots(figsize=(size, size * 0.85))
        mask = np.triu(np.ones_like(corr_plot, dtype=bool))
        sns.heatmap(
            corr_plot, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            annot=(n <= 20), fmt=".2f",
            linewidths=0.3 if n <= 30 else 0, square=True, ax=ax,
        )
        ax.set_title(f"{group_name} — Correlation matrix{subtitle}", fontsize=11)
        plt.tight_layout()
        plt.show()
    else:
        print("  No columns above threshold to plot.")

    if len(high_corr) > 0:
        print(f"\n  Top {min(top_n, len(high_corr))} pairs  |r| >= {threshold}:")
        print(high_corr.head(top_n)[["var1", "var2", "r"]].to_string(index=False))
    else:
        print(f"\n  No pairs with |r| >= {threshold} found.")


def _jaccard_analysis(df, cols, group_name, threshold=0.2, top_n=20):
    """
    Jaccard similarity matrix for sparse binary columns.
    J(A,B) = n(A=1 AND B=1) / n(A=1 OR B=1)
    Ignores (0,0) rows — no sparsity artefact.
    """
    data = (
        df[cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
    )
    data = data.loc[:, (data == 1).any()]
    cols_valid = data.columns.tolist()
    n = len(cols_valid)

    if n < 2:
        print(f"\n{group_name}: not enough columns with at least one 1.")
        return

    arr    = data.values.astype(np.float32)
    both   = arr.T @ arr
    a_sum  = (arr == 1).sum(axis=0).reshape(-1, 1)
    b_sum  = (arr == 1).sum(axis=0).reshape(1, -1)
    either = a_sum + b_sum - both
    with np.errstate(invalid="ignore", divide="ignore"):
        jac = np.where(either > 0, both / either, 0.0)
    np.fill_diagonal(jac, 1.0)
    jaccard = pd.DataFrame(jac, index=cols_valid, columns=cols_valid)

    pairs = (
        jaccard.where(np.triu(np.ones(jaccard.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ["var1", "var2", "jaccard"]
    pairs = pairs.sort_values("jaccard", ascending=False)
    high_pairs = pairs[pairs["jaccard"] >= threshold]

    print(f"\n{'='*60}")
    print(f"  {group_name}  ({n} features)")
    print(f"  Pairs with Jaccard >= {threshold}: {len(high_pairs)}")
    print(f"{'='*60}")

    size = max(6, n * 0.45)
    fig, ax = plt.subplots(figsize=(size, size * 0.85))
    mask = np.triu(np.ones_like(jaccard, dtype=bool))
    sns.heatmap(
        jaccard, mask=mask, cmap="YlOrRd", vmin=0, vmax=1,
        annot=(n <= 20), fmt=".2f", linewidths=0.5, square=True, ax=ax,
    )
    ax.set_title(f"{group_name}\nJaccard similarity  [J = n(A=1∩B=1) / n(A=1∪B=1)]", fontsize=11)
    plt.tight_layout()
    plt.show()

    if len(high_pairs) > 0:
        print(f"\n  Top {min(top_n, len(high_pairs))} pairs  Jaccard >= {threshold}:")
        print(f"  {'var1':<32}  {'var2':<32}  {'J':>6}  {'n_both':>7}  {'n_A':>6}  {'n_B':>6}")
        print(f"  {'─'*32}  {'─'*32}  {'─'*6}  {'─'*7}  {'─'*6}  {'─'*6}")
        for _, row in high_pairs.head(top_n).iterrows():
            a, b   = row["var1"], row["var2"]
            n_both = int(((data[a] == 1) & (data[b] == 1)).sum())
            n_a    = int((data[a] == 1).sum())
            n_b    = int((data[b] == 1).sum())
            print(f"  {a:<32}  {b:<32}  {row['jaccard']:>6.3f}  {n_both:>7,}  {n_a:>6,}  {n_b:>6,}")
    else:
        print(f"\n  No pairs with Jaccard >= {threshold} found.")

## General features

In [ ]:
_corr_analysis(df_stroke_general, other_cols, "Other features",        threshold=0.3)

## Past history

In [ ]:
_corr_analysis(df_stroke_reduced, hist_cols,  "Past history (hist_)",  threshold=0.3)

In [ ]:
df_with_history = df_stroke_general[df_stroke_general["has_history"] == 1].copy()
print(f"Patients with has_history=1: {len(df_with_history):,} / {len(df_stroke_general):,} ({len(df_with_history)/len(df_stroke_general)*100:.1f}%)\n")

_jaccard_analysis(df_with_history, hist_cols, "Past history (hist_) — has_history=1 only", threshold=0.2)

## Literal reason

In [ ]:
_corr_analysis(df_stroke_reduced, lr_cols,    "Literal reason (lr_)",  threshold=0.3)

In [ ]:
_jaccard_analysis(df_stroke_general, lr_cols, "Literal reason (lr_) — all patients", threshold=0.2)

## Medication

In [ ]:
_corr_analysis(df_stroke_reduced, atc_cols,   "Medication (atc)",       threshold=0.3)

In [ ]:
df_with_med = df_stroke_general[df_stroke_general["has_med"] == 1].copy()
print(f"Patients with has_med=1: {len(df_with_med):,} / {len(df_stroke_general):,} ({len(df_with_med)/len(df_stroke_general)*100:.1f}%)\n")

_jaccard_analysis(df_with_med, atc_cols, "Medication (atc) — has_med=1 only", threshold=0.2)